In [ ]:
# XNET 

# !pip -q install transformers datasets evaluate scikit-learn pandas numpy matplotlib

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from datasets import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification

print("Libraries loaded for XNET approach.")

In [ ]:
# load dataset and build cross-encoder input

train_df = pd.read_csv("../data/train.csv")
val_df = pd.read_csv("../data/val.csv")
test_df = pd.read_csv("../data/test.csv")

def safe_text(x):
    if pd.isna(x):
        return ""
    return str(x).strip()

# 🔥 Key difference: structured input for XNet
def build_xnet_input(row):
    src = safe_text(row["text_src"])
    tgt = safe_text(row["text_tgt"])

    return f"Source: {src} [SEP] Edited: {tgt}"

train_df["input_text"] = train_df.apply(build_xnet_input, axis=1)
val_df["input_text"] = val_df.apply(build_xnet_input, axis=1)
test_df["input_text"] = test_df.apply(build_xnet_input, axis=1)

# label mapping
labels = train_df["label"].unique()
label2id = {label: i for i, label in enumerate(labels)}
id2label = {i: label for label, i in label2id.items()}

train_df["label_id"] = train_df["label"].map(label2id)
val_df["label_id"] = val_df["label"].map(label2id)
test_df["label_id"] = test_df["label"].map(label2id)

print("Train shape:", train_df.shape)
print("Validation shape:", val_df.shape)
print("Test shape:", test_df.shape)

print("\nSample XNET input:")
print(train_df["input_text"].iloc[0])

print("\nLabel mapping:")
print(label2id)

In [ ]:
# convert to HuggingFace datasets

from datasets import Dataset

train_xnet = Dataset.from_pandas(train_df[["input_text", "label_id"]])
val_xnet = Dataset.from_pandas(val_df[["input_text", "label_id"]])
test_xnet = Dataset.from_pandas(test_df[["input_text", "label_id"]])

print("Train dataset:", train_xnet)
print("Validation dataset:", val_xnet)
print("Test dataset:", test_xnet)

In [ ]:
#  load tokenizer and model

from transformers import AutoTokenizer, AutoModelForSequenceClassification

xnet_model_name = "distilbert-base-uncased"

xnet_tokenizer = AutoTokenizer.from_pretrained(xnet_model_name)

xnet_model = AutoModelForSequenceClassification.from_pretrained(
    xnet_model_name,
    num_labels=len(label2id),
    id2label=id2label,
    label2id=label2id
)

print("XNET model loaded:", xnet_model_name)
print("Number of labels:", len(label2id))

In [ ]:
# tokenize data

max_length = 128

def tokenize_xnet(example):
    return xnet_tokenizer(
        example["input_text"],
        padding="max_length",
        truncation=True,
        max_length=max_length
    )

train_xnet_tokenized = train_xnet.map(tokenize_xnet, batched=True)
val_xnet_tokenized = val_xnet.map(tokenize_xnet, batched=True)
test_xnet_tokenized = test_xnet.map(tokenize_xnet, batched=True)

print("Tokenization completed.")
print(train_xnet_tokenized[0])

In [ ]:
#  set format for training

train_xnet_tokenized = train_xnet_tokenized.rename_column("label_id", "labels")
val_xnet_tokenized = val_xnet_tokenized.rename_column("label_id", "labels")
test_xnet_tokenized = test_xnet_tokenized.rename_column("label_id", "labels")

columns = ["input_ids", "attention_mask", "labels"]

train_xnet_tokenized.set_format(type="torch", columns=columns)
val_xnet_tokenized.set_format(type="torch", columns=columns)
test_xnet_tokenized.set_format(type="torch", columns=columns)

print("XNET dataset ready.")
print(train_xnet_tokenized[0])

In [ ]:
# metrics

import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

def compute_xnet_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=1)

    accuracy = accuracy_score(labels, predictions)
    precision, recall, f1, _ = precision_recall_fscore_support(
        labels,
        predictions,
        average="weighted",
        zero_division=0
    )

    return {
        "accuracy": accuracy,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }

print("XNET metrics ready.")

In [ ]:
# training

import torch
from transformers import TrainingArguments, Trainer

xnet_training_args = TrainingArguments(
    output_dir="./xnet_results",
    eval_strategy="epoch",
    save_strategy="epoch",
    logging_strategy="epoch",
    num_train_epochs=3,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    learning_rate=2e-5,
    weight_decay=0.01,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    report_to="none",
    fp16=torch.cuda.is_available()
)

xnet_trainer = Trainer(
    model=xnet_model,
    args=xnet_training_args,
    train_dataset=train_xnet_tokenized,
    eval_dataset=val_xnet_tokenized,
    compute_metrics=compute_xnet_metrics
)

xnet_trainer.train()

In [ ]:
# evaluate on test set

xnet_test_results = xnet_trainer.evaluate(test_xnet_tokenized)
print(xnet_test_results)

In [ ]:
# predictions

xnet_pred_output = xnet_trainer.predict(test_xnet_tokenized)

print("Prediction completed.")
print("Shape:", xnet_pred_output.predictions.shape)

In [ ]:
# convert predictions

import numpy as np

xnet_predictions = np.argmax(xnet_pred_output.predictions, axis=1)
xnet_true_labels = xnet_pred_output.label_ids

xnet_pred_names = [id2label[i] for i in xnet_predictions]
xnet_true_names = [id2label[i] for i in xnet_true_labels]

print("Sample predictions:")
for i in range(5):
    print("Pred:", xnet_pred_names[i], "| True:", xnet_true_names[i])

In [ ]:
# classification report

from sklearn.metrics import classification_report

print(classification_report(
    xnet_true_names,
    xnet_pred_names,
    labels=list(label2id.keys()),
    zero_division=0
))

In [ ]:
#  confusion matrix

from sklearn.metrics import confusion_matrix
import pandas as pd
import matplotlib.pyplot as plt

label_list = list(label2id.keys())

cm = confusion_matrix(xnet_true_names, xnet_pred_names, labels=label_list)

cm_df = pd.DataFrame(cm, index=label_list, columns=label_list)
print(cm_df)

plt.figure(figsize=(8,6))
plt.imshow(cm)
plt.title("XNET Confusion Matrix")
plt.colorbar()
plt.xticks(range(len(label_list)), label_list, rotation=45)
plt.yticks(range(len(label_list)), label_list)
plt.xlabel("Predicted")
plt.ylabel("True")
plt.tight_layout()
plt.show()

In [ ]:
# save model

from google.colab import drive
drive.mount('/content/drive')

save_path = "/content/drive/MyDrive/xnet_model"

xnet_trainer.save_model(save_path)
xnet_tokenizer.save_pretrained(save_path)

print("Model saved at:", save_path)